# 问题 4：影响关系是否真的有音乐特征证据？

检验 `influence_data` 里被记录的 influencer 是否真的影响了 follower，并找出哪些特征更「传染」。

**方法：**
1. 取双方都有画像的影响边为样本
2. 为每条边匹配一个「未记录关系」对照（同流派随机艺术家）
3. 效应量 `ΔS = 记录关系相似度 − 对照相似度`
4. 特征级 `Δ_k = 对照特征差 − 记录关系特征差`（正值 = 该特征更接近 = 更传染）
5. 排除共同合作歌曲的影响

注意：对照措辞是「未记录关系」而非「确定无影响」。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['figure.dpi'] = 100

artists = pd.read_csv('output/artists.csv', dtype=str)
edges = pd.read_csv('output/edges.csv', dtype=str)
song_artist = pd.read_csv('output/song_artist.csv', dtype=str)

FEATURES = ['danceability', 'energy', 'valence', 'tempo', 'loudness',
            'acousticness', 'instrumentalness', 'liveness', 'speechiness', 'duration_ms']
for f in FEATURES:
    artists[f] = pd.to_numeric(artists[f], errors='coerce')

# 标准化画像矩阵
Zmat = ((artists[FEATURES] - artists[FEATURES].mean()) / artists[FEATURES].std()).values
valid = artists[FEATURES].notna().all(axis=1).values
artists_v = artists[valid].reset_index(drop=True)
Zmat_v = Zmat[valid]

zmap = dict(zip(artists_v['artist_id'], Zmat_v))
genre_map = dict(zip(artists['artist_id'], artists['genre']))
genre_artists = artists.groupby('genre')['artist_id'].apply(list).to_dict()

print('有完整画像的艺术家:', len(zmap))

## 1. 构建影响样本 + 匹配对照 + 共同作品标记

In [ ]:
sub_edges = edges[(edges['influencer_id'].isin(zmap)) & (edges['follower_id'].isin(zmap))].copy()
inf_of = sub_edges.groupby('follower_id')['influencer_id'].apply(set).to_dict()
artist_songs = song_artist.groupby('artist_id')['song_index'].apply(set).to_dict()

rng = np.random.default_rng(42)
real_sims, ctrl_sims, real_diff, ctrl_diff, collabs = [], [], [], [], []

for i, f in zip(sub_edges['influencer_id'].values, sub_edges['follower_id'].values):
    zi = zmap[i]; zf = zmap[f]
    real_sims.append(1.0 / (1.0 + np.linalg.norm(zi - zf)))
    real_diff.append(np.abs(zi - zf))

    # 匹配对照：f 所在流派里随机一个未记录影响者
    pool = genre_artists.get(genre_map.get(f), [])
    c = None
    if len(pool) > 1:
        infset = inf_of.get(f, set())
        for _ in range(50):
            cand = pool[rng.integers(len(pool))]
            if cand != f and cand not in infset and cand in zmap:
                c = cand; break
    if c is None:
        ctrl_sims.append(np.nan); ctrl_diff.append(np.full(len(FEATURES), np.nan))
    else:
        zc = zmap[c]
        ctrl_sims.append(1.0 / (1.0 + np.linalg.norm(zc - zf)))
        ctrl_diff.append(np.abs(zc - zf))

    # 是否共同参与过某首歌
    collabs.append(len(artist_songs.get(i, set()) & artist_songs.get(f, set())) > 0)

real_sims = np.array(real_sims); ctrl_sims = np.array(ctrl_sims)
real_diff = np.array(real_diff); ctrl_diff = np.array(ctrl_diff)
collabs = np.array(collabs)

print('有效影响边:', len(sub_edges))
print('其中有共同合作歌曲的边:', collabs.sum())

## 2. 效应量 ΔS 与显著性

In [ ]:
ok = ~np.isnan(ctrl_sims)
deltaS = real_sims[ok].mean() - ctrl_sims[ok].mean()
t, p = stats.ttest_rel(real_sims[ok], ctrl_sims[ok])

print(f'记录关系平均相似度 = {real_sims[ok].mean():.4f}')
print(f'对照关系平均相似度 = {ctrl_sims[ok].mean():.4f}')
print(f'ΔS = {deltaS:.4f}  (配对 t 检验 p = {p:.2e})')

plt.figure(figsize=(8, 5))
plt.hist(ctrl_sims[ok], bins=40, alpha=0.6, label='Unrecorded (control)', color='#55A868')
plt.hist(real_sims[ok], bins=40, alpha=0.6, label='Recorded influence', color='#C44E52')
plt.xlabel('Similarity S = 1/(1+d)')
plt.ylabel('Count')
plt.title('Recorded influence vs matched control')
plt.legend()
plt.tight_layout(); plt.show()

## 3. 哪些特征更「传染」

`Δ_k = 对照平均特征差 − 记录关系平均特征差`，正值越大说明该特征在记录关系中更接近。

In [ ]:
delta_k = ctrl_diff[ok].mean(axis=0) - real_diff[ok].mean(axis=0)

order = np.argsort(delta_k)[::-1]
plt.figure(figsize=(9, 5))
plt.bar(np.array(FEATURES)[order], delta_k[order], color='#4C72B0')
plt.axhline(0, color='black', lw=0.8)
plt.ylabel(r'$\Delta_k$ (control − recorded)')
plt.title('Feature-level contagion (larger = more contagious)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

print('按传染性排序的特征:')
for f, d in zip(np.array(FEATURES)[order], delta_k[order]):
    print(f'  {f:18s} {d:+.4f}')

## 4. 排除共同作品后的稳健性

共同参与同一首歌会直接抬高相似度，排除后再看 ΔS 是否仍成立。

In [ ]:
nocollab = ok & ~collabs
deltaS_nc = real_sims[nocollab].mean() - ctrl_sims[nocollab].mean()
t_nc, p_nc = stats.ttest_rel(real_sims[nocollab], ctrl_sims[nocollab])

print(f'排除共同作品后: ΔS = {deltaS_nc:.4f} (p = {p_nc:.2e})')
print(f'对照: 含共同作品时 ΔS = {deltaS:.4f}')
print('→ 若排除后 ΔS 仍显著为正，说明影响关系有超越共同合作的独立证据')

## 5. 结论

- 若 ΔS 显著 > 0，说明**被记录的影响关系在音乐特征上确实比随机对照更相似**，即「influencer 确实影响了 follower」有特征证据。
- 特征级 Δ_k 揭示哪些音频特征更容易在影响关系中「同步」（传染性）。
- 排除共同作品后结论仍成立，则证据更可靠；反之则相似度可能主要来自合作而非影响。
- 局限：即使控制流派/年代，仍可能是共同环境或未观测因素；影响也可能发生在歌词/技巧等未测量维度。